## PDF Reader

### Restaurant Reader

In [ ]:
!pip install langchain langchain-community langchain-google-genai

In [5]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

print("Key loaded:", bool(key))

Key loaded: True


In [6]:
import os
#google ai studio
os.environ["GOOGLE_API_KEY"] = key

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    temperature = 0
) #when a question is asked, for the answer to not change temp = 0 is used.token limit can also be set.timeout.number of times to regenerate.

In [ ]:
question = input("Ask something: ")

response = llm.invoke(question)
answer = response.content

if isinstance(answer, list):
      answer = "\n".join(
        item["text"]
        for item in response.content
        if isinstance(item,dict) and item.get("type") == "text"
    )

print(answer)

In [ ]:
!pip install pypdf

In [ ]:
import gradio as gr
from pypdf import PdfReader

from langchain_google_genai import ChatGoogleGenerativeAI
# 1. Read restaurant pdf
reader = PdfReader("/content/Resturaunt Q&A.pdf")

restaurant_info = ""
for page in reader.pages:
  text = page.extract_text()
  if text:
    restaurant_info += text+"\n"

def chatbot(message, history):
  prompt = f"""
  You are restaurant customer support assisstant.
  Use the following restaurant information to answer the customers question.
  Restaurant information:
  {restaurant_info}
  Customer Question:
  {message}
  Answer the customer clearly and politely.
  If the information is not available in the restaurant information, say thst you don't have the information
"""
  response = llm.invoke(prompt)
  return response.content

demo = gr.ChatInterface(
    fn = chatbot,
    title = "Restaurant Support Bot",
    description = "Ask question about our restaurant"
)
demo.launch()


### Resume Reader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [ ]:
# !pip install -U langchain-community pypdf

In [ ]:
#1. load pdf
loader = PyPDFLoader("/content/resume.pdf")
documents = loader.load()

In [ ]:
from langchain.tools import tool

In [ ]:
#2. create a simple tool for the agent
@tool
def read_pdf(question: str) -> str:
    """Answer a question using the PDF content."""
    text = "\n".join(doc.page_content for doc in documents)
    return text

In [ ]:
# create agent
agent = create_agent(
    model = llm,
    tools = [read_pdf],
    system_prompt = """
    You are a PDF reader agent.
    Use the read_pdf tool to find information from the PDF.
    Answer only using information from the PDF.
    If the answer id not available in the PDF, say: 'I could not find this information in the PDF.'
    """
)

In [ ]:
#5. Ask question
question = input("Ask something about PDF: ")

result = agent.invoke({
    "messages" : [
        {"role" : "user", "content": question}
    ]
})

#get final answer
answer = result["messages"][-1].content

# if gemini return a list, extract the text
if isinstance(answer, list):
  answer = "\n".join(
      item["text"]
      for item in answer
      if isinstance(item, dict) and item.get("type") == "text"
  )

print("PDF AGENT ANSWER:")
print(answer)

## Calculator Agent

In [60]:
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


@tool
def addition(a: float, b: float):
    """Add two numbers."""
    return a + b


@tool
def subtraction(a: float, b: float):
    """Subtract the second number from the first number."""
    return a - b


@tool
def multiplication(a: float, b: float):
    """Multiply two numbers."""
    return a * b


@tool
def division(a: float, b: float):
    """Divide the first number by the second number."""
    if b == 0:
        return "Cannot divide by zero"
    return a / b


tool= [
    addition,
    subtraction,
    multiplication,
    division
]

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature = 0
)

agent = create_agent(
    model = llm ,
    tools = tool ,
    system_prompt="Do the calulations, if the input is not numerical based or cannot be evaluated say: Invalid input"
)


In [ ]:
question = "1+0998"
res = agent.invoke({
    "messages":[{"role":"user" , "content":question}]
})
res["messages"][-1].content

## Calculator tool

In [64]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.
    Use this tool when the user asks for a mathematical calculation.
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [ ]:
question = "1+998"
res = calculator.invoke({
    "expression": question
})
print(res)

## Placement Assisstant

In [77]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [78]:
@tool
def get_student(student_id: str) -> str:
    """
    Get the placement profile of a student using their student ID.
    Use this tool whenever the user asks for student placement information.
    """

    students = {
        "22CS045": {
            "name": "Arun",
            "branch": "CSE",
            "cgpa": 8.7,
            "skills": ["Python", "SQL", "C++"]
        },

        "22CS046": {
            "name": "Priya",
            "branch": "AIML",
            "cgpa": 9.1,
            "skills": ["Python", "Machine Learning", "SQL"]
        }
    }

    if student_id not in students:
        return "Student not found."

    return str(students[student_id])

In [79]:
@tool
def search_jobs(skill: str) -> str:
    """
    Search available placement opportunities based on a student's skill.
    Use this when the user asks about jobs, companies, or placement opportunities.
    """

    jobs = {
        "python": [
            "TCS - Python Developer",
            "Infosys - Python Developer",
            "Zoho - Backend Developer"
        ],
        "sql": [
            "TCS - Database Developer",
            "Accenture - Data Analyst"
        ],
        "machine learning": [
            "Zoho - ML Intern",
            "TCS - AI Engineer"
        ]
    }

    skill = skill.lower()

    if skill not in jobs:
        return f"No jobs found for {skill}."

    return str(jobs[skill])

In [80]:
@tool
def check_eligibility(cgpa: float) -> str:
    """
    Check whether a student satisfies the minimum CGPA requirement for placement.
    Use this when the user asks whether a student is eligible for placement.
    """

    if cgpa >= 8.0:
        return "Eligible for companies requiring a CGPA of 8.0 or above."

    return "Not eligible for companies requiring a CGPA of 8.0."

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature = 0
)
agent = create_agent(
    model=llm,
    tools=[get_student, search_jobs, check_eligibility],
    system_prompt="""
    You are a placement assistant.

    When the user asks for student placement information,
    use the tool.

    Do not invent student information.
    """
)
question = input("Ask question about placement: ")
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})
print(result["messages"][-1].content)